<a href="https://colab.research.google.com/github/AbhishekShah272002/Deep-Learning-and-NLP/blob/main/daily_minimum_temperatures_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

In [ ]:
data = pd.read_csv("daily-minimum-temperatures.csv")

In [ ]:
print("Columns in dataset:", data.columns.tolist())
display(data.head())

Columns in dataset: ['Date', 'Daily minimum temperatures in Melbourne, Australia, 1981-1990', 'Unnamed: 2']


,Date,"Daily minimum temperatures in Melbourne, Australia, 1981-1990",Unnamed: 2
0,01-01-1981,20.7,NaN
1,02-01-1981,17.9,NaN
2,03-01-1981,18.8,NaN
3,04-01-1981,14.6,NaN
4,05-01-1981,15.8,NaN


In [ ]:
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
import numpy as np
import joblib


scaler = MinMaxScaler(feature_range=(0, 1))
scaled_all = scaler.fit_transform(temperatures.reshape(-1, 1))

# Save the scaler
joblib.dump(scaler, 'scaler.pkl')
print("Scaler created and saved as scaler.pkl")


X_train, y_train = [], []
for i in range(len(scaled_all) - 7):
    X_train.append(scaled_all[i:i+7])
    y_train.append(scaled_all[i+7])
X_train, y_train = np.array(X_train), np.array(y_train)


model = Sequential([
    LSTM(50, activation='relu', input_shape=(7, 1)),
    Dense(1)
])
model.compile(optimizer='adam', loss='mse')
print("Training a quick LSTM model to use for predictions...")
model.fit(X_train, y_train, epochs=5, batch_size=16, verbose=1)


model.save('model.h5')
print("Model trained and saved as model.h5")

Scaler created and saved as scaler.pkl


/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Training a quick LSTM model to use for predictions...
Epoch 1/5
228/228 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 0.0289
Epoch 2/5
228/228 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0102
Epoch 3/5
228/228 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0098
Epoch 4/5
228/228 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0096
Epoch 5/5
228/228 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0092


Model trained and saved as model.h5


In [ ]:
print("Temperatures successfully mapped and loaded.")

Temperatures successfully mapped and loaded.


In [ ]:
temp_col = [col for col in data.columns if 'temp' in col.lower() or 'value' in col.lower()][0]


clean_data = data.dropna(subset=[temp_col]).copy()
clean_data[temp_col] = pd.to_numeric(clean_data[temp_col], errors='coerce')
clean_data = clean_data.dropna(subset=[temp_col])

temperatures = clean_data[temp_col].values
print(f"Successfully extracted {len(temperatures)} temperature records using column: '{temp_col}'")

Successfully extracted 3647 temperature records using column: 'Daily minimum temperatures in Melbourne, Australia, 1981-1990'


In [ ]:
last_seven_days = temperatures[-7:]
print("Last seven days of temperatures:", last_seven_days)

Last seven days of temperatures: [12.9 14.6 14.  13.6 13.5 15.7 13. ]


In [ ]:
scaled_data = scaler.transform(last_seven_days.reshape(-1, 1))

In [ ]:
X = scaled_data.reshape(1, 7, 1)

In [ ]:
scaled_prediction = model.predict(X, verbose=0)

In [ ]:
next_day_temperature = scaler.inverse_transform(scaled_prediction.reshape(-1, 1))[0, 0]

In [ ]:
input_data = pd.DataFrame({
    "Day": clean_data["Date"].iloc[-7:].values if "Date" in clean_data.columns else clean_data.index[-7:] + 1,
    "Temperature (°C)": last_seven_days
})
display(input_data)

,Day,Temperature (°C)
0,25-12-1990,12.9
1,26-12-1990,14.6
2,27-12-1990,14.0
3,28-12-1990,13.6
4,29-12-1990,13.5
5,30-12-1990,15.7
6,31-12-1990,13.0


In [ ]:
prediction_result = pd.DataFrame({
    "Prediction": ["Next day"],
    "Temperature (°C)": [next_day_temperature]
})
display(prediction_result)

,Prediction,Temperature (°C)
0,Next day,13.616649
